# OVITO DXA：分组检查与性能比较

面向已有 OVITO Python 环境的使用者。本示例只分析随仓库生成的完整 BCC/FCC 晶体，不包含私人数据。

逐个运行单元格：确认解释器 → 生成合成输入 → 检查分组与设置 → 串行基线与性能比较 → 全量示例与续算。使用真实数据时，应先向数据提供者确认分组、晶格、单位、边界和输出档次。

In [ ]:
from pathlib import Path  # 使用跨平台路径。
import sys  # 获取当前 Notebook 内核的真实解释器。
import json  # 查看实际任务清单。
import subprocess  # 通过独立解释器启动分析脚本。
from datetime import datetime, timezone  # 创建本次独立示例目录。
from IPython.display import HTML, display  # 在 Notebook 中显示性能报告。
root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "tests/make_examples.py").is_file()), None)  # 从仓库根或 examples 目录启动。
if root is None:  # 避免在错误目录猜测文件路径。
    raise RuntimeError("请从仓库根目录或 examples 目录打开此 Notebook。")  # 清楚指出当前一步。
python = sys.executable  # 不调用可能属于其他环境的 python 别名。
subprocess.run([python, "-c", "import ovito, psutil; print(ovito.version_string)"], check=True)  # 先确认内核环境具备依赖。
print("解释器：", python)  # 展示实际执行环境。

## 生成可运行示例

每次创建新的目录，不覆盖旧结果。示例中的晶格、单位及全周期边界由生成器明确定义；它们不是实际研究数据的默认设置。

In [ ]:
example_dir = root / "examples" / ("notebook_generated_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f"))  # 创建唯一结果位置。
subprocess.run([python, str(root / "tests/make_examples.py"), "--output", str(example_dir)], check=True)  # 生成合成数据与完整清单。
config = example_dir / "task.json"  # 使用已生成的两组独立数据清单。
settings = json.loads(config.read_text(encoding="utf-8"))  # 读取真实配置。
print(json.dumps(settings, ensure_ascii=False, indent=2))  # 检查分组、单位与分析参数。

## 只读检查输入

本次检查同一条三帧 BCC 轨迹；不会执行 DXA。若一次提供许多文件，检查结果仅是候选分组，仍需确定这些文件是否属于同一轨迹。

In [ ]:
inspector = root / "skills/ovito-dxa/scripts/inspect_inputs.py"  # 定位只读检查器。
inspection = subprocess.run([python, str(inspector), "--paths", str(example_dir / "bcc.dump"), "--all-frames"], check=True, capture_output=True, text=True, encoding="utf-8")  # 检查所有示例帧的可读取性。
print(inspection.stdout)  # 查看真实检查报告与待确认项。

## 串行基线

数据分组与科学设置在上方清单中可见。此单元运行代表帧的优化串行实现，不代表用户旧脚本的实测成绩。

In [ ]:
runner = root / "skills/ovito-dxa/scripts/run_dxa.py"  # 定位可导入 worker 的独立脚本。
subprocess.run([python, str(runner), "--config", str(config), "--mode", "baseline"], check=True)  # 独立进程运行串行基线。

## 比较性能与一致性

比较读取、初始化、计算、通信及保存耗时。小数据常因进程启动成本而没有加速；报告会保留实际结果。只有结果一致的候选才能成为推荐配置。

In [ ]:
subprocess.run([python, str(runner), "--config", str(config), "--mode", "benchmark"], check=True)  # 实测代表帧的串行与并行候选。
report = example_dir / "results/task/benchmark/report.html"  # 定位本次真实 HTML 报告。
display(HTML(report.read_text(encoding="utf-8")))  # 直接展示耗时、内存和一致性。

## 全量示例与续算

这里的全量任务仅包含三个合成 BCC 帧与一个合成 FCC 构型。真实研究数据应在阅读样本报告、确认资源配置后再执行全量。续算会检查来源与科学设置，不把已有文件直接当成成功。

In [ ]:
subprocess.run([python, str(runner), "--config", str(config), "--mode", "run"], check=True)  # 执行全部已选择的合成帧。
subprocess.run([python, str(runner), "--config", str(config), "--mode", "resume"], check=True)  # 验证已完成结果可安全复用。
print("结果目录：", example_dir / "results/task/run")  # 提供本次输出的真实位置。

## 自检练习

`bcc.dump` 和 `bcc.0.dump`、`bcc.10.dump`、`bcc.20.dump` 是四组独立工况吗？

答案：不是。生成器将同一条三帧轨迹保存为一个多帧文件和三个分帧文件。`sequence.json` 明确了后一种表示。真实输入仅凭文件名无法确定这种对应关系。

常见误区：`%run` 不是并行开关；并行来自脚本内部的进程池。单构型不应伪造帧级任务，零线长也不能等同读取失败。若扩展溶质分析，需要另外确认线点/线段距离和周期边界。